This notebook cleans and verifys that the parquets for Humboldt, Chico, Somona, Sac, San Fran, San Berninadino, Pomona are of the following schema:
- Name (First Last)
- School
- Department
- Title
- Major (null if they don't have)

In [114]:
humboldt_data = "../Data/humboldt_faculty.parquet"
chico_data = "../Data/chico_faculty.parquet"
sonoma_data = "../Data/sonoma_faculty.parquet"
sac_data = "../Data/csusac_faculty.parquet"
sfsu_data = "../Data/sfsu_faculty.parquet"
sanbern_data = "../Data/csusb_faculty.parquet"
pomona_data = "../Data/cpp_faculty.parquet"
dh_data = "../Data/csudh_faculty.parquet"

In [26]:
import pandas as pd
import duckdb
import numpy as np
from pathlib import Path

In [27]:
chico_df = pd.read_parquet(chico_data)
chico_df.head(5)

,name,title,department,school,year
0,Dawn M Abel,Lecturer,Nursing,"California State University, Chico",2013
1,Annie Adamian,Associate Professor,Education,"California State University, Chico",2014
2,Nora A Aguilar McKay,Lecturer,Education,"California State University, Chico",2014
3,Kelsey R Aguire,Lecturer,Nursing,"California State University, Chico",2025
4,Kirk D Aiken,Professor,Marketing,"California State University, Chico",2013


In [31]:
chico_df['title'].unique().tolist()

['Lecturer',
 'Associate Professor',
 'Professor',
 'Assistant Professor',
 'Chair',
 'Grsf If 12',
 'Director',
 'Vice Chair',
 'Program Coordinator',
 'Instructor',
 'Assistant Director',
 'Emeritus']

- Name needs to be cleaned (remove middle initial)

In [ ]:


chico_df["name"] = (
    chico_df["name"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.replace(r"(?<=\S) (?:[A-Za-z]\.? )+(?=\S)", " ", regex=True)
    .str.replace(r"(?i),? (?:Jr\.?|Sr\.?|II|III|IV)$", "", regex=True)
)

chico_output = Path(chico_data).with_name("chico_cleaned.parquet")
chico_df.to_parquet(chico_output, index=False)
chico_df.head(5)

,name,title,department,school,year
0,Dawn Abel,Lecturer,Nursing,"California State University, Chico",2013
1,Annie Adamian,Associate Professor,Education,"California State University, Chico",2014
2,Nora Aguilar McKay,Lecturer,Education,"California State University, Chico",2014
3,Kelsey Aguire,Lecturer,Nursing,"California State University, Chico",2025
4,Kirk Aiken,Professor,Marketing,"California State University, Chico",2013


In [113]:
chico_df.shape

(1468, 5)

In [48]:
sac_df = pd.read_parquet(sac_data)
sac_df.head(5)

,name,title,department,school,start_date
0,"ABADI, MASOUD",Assistant Professor of Civil Engineering,Civil Engineering,"California State University, Sacramento",2018
1,"ABAH, TERESA",Assistant Professor of Gerontology,Gerontology,"California State University, Sacramento",2020
2,"ABENDROTH, KATHLEEN","Maryjane Rees Language, Speech and Hearing Cen...",CSAD,"California State University, Sacramento",2022
3,"ABULHASSAN, YOUSIF A.",Associate Professor of Public Health,Public Health,"California State University, Sacramento",2022
4,"ACEVES-AZUARA, ITZEL",Assistant Professor of Psychology,Psychology,"California State University, Sacramento",2022


In [39]:
sac_df['title'].nunique()

337

clean titles:

In [52]:
# Replace df with your dataframe name.
titles = (
    sac_df["title"]
    .astype("string")
    .str.strip()
    .str.replace(r"\bAssoiciate\b", "Associate", regex=True, case=False)
    .str.replace(r"\bProfesssor\b", "Professor", regex=True, case=False)
)

# Put more specific titles before their shorter versions.
pattern = (
    r"(?i)\b("
    r"Senior Assistant Librarian|"
    r"Assistant Professor|"
    r"Associate Professor|"
    r"Professor|"
    r"Full-Time Lecturer|"
    r"Lecturer|"
    r"Librarian"
    r")\b"
)

matched = titles.str.extract(pattern, expand=False)
sac_df["title_clean"] = matched.str.title().fillna(titles)
# Keep unmatched titles for review instead of dropping them.
sac_df["title"] = sac_df["title_clean"].fillna(sac_df["title"])
sac_df["title"] = sac_df["title"].replace(
    "AssociateProfessor of Physical Therapy",
    "Associate Professor"
)
sac_df.drop(columns=["title_clean"], inplace=True)

In [50]:
sac_df['title'].unique().tolist()

['Assistant Professor',
 'Full-Time Lecturer',
 'Associate Professor',
 'Professor',
 'Librarian',
 'Lecturer',
 'Senior Assistant Librarian',
 'MPP; Associate Dean for Collections & Discovery, University Library',
 'MPP; Dean, University Library',
 'MPP; Associate Dean for Academic Services, University Library',
 'Full-time Temporary']

Clean Name

In [53]:
sac_df["name"] = (
    sac_df["name"]
    .str.replace(r"^\s*([^,]+),\s*(.+?)\s*$", r"\2 \1", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.replace(r"(?<=\S) (?:[A-Za-z]\.? )+(?=\S)", " ", regex=True)
    .str.title()
)
sac_output = Path(sac_data).with_name("sac_cleaned.parquet")
sac_df.to_parquet(sac_output, index=False)
sac_df.head(5)

,name,title,department,school,start_date
0,Masoud Abadi,Assistant Professor,Civil Engineering,"California State University, Sacramento",2018
1,Teresa Abah,Assistant Professor,Gerontology,"California State University, Sacramento",2020
2,Kathleen Abendroth,Full-Time Lecturer,CSAD,"California State University, Sacramento",2022
3,Yousif Abulhassan,Associate Professor,Public Health,"California State University, Sacramento",2022
4,Itzel Aceves-Azuara,Assistant Professor,Psychology,"California State University, Sacramento",2022


In [112]:
sac_df.shape

(807, 5)

In [11]:
sfsu_df = pd.read_parquet(sfsu_data)
sfsu_df.head(5)

,name,title,department,school,start_date
0,Miguel Abad,Assistant Professor in Child and Adolescent De...,Child and Adolescent Development,San Francisco State University,2022
1,Bárbara I. Abadía-Rexach,Associate Professor in Latina/Latino Studies,Latina/Latino Studies,San Francisco State University,2020
2,Shadee Abdi,Associate Professor in Communication Studies,Communication Studies,San Francisco State University,2019
3,Rabab Abdulhadi,Associate Professor in Race and Resistance Stu...,Race and Resistance Studies,San Francisco State University,2007
4,Priyanvada Abeywickrama,Professor in English Language and Literature,English Language and Literature,San Francisco State University,2007


Name and title need to be cleaned

In [56]:
sfsu_df["name"] = (
    sfsu_df["name"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.replace(r"(?<=\S) (?:[A-Za-z]\.? )+(?=\S)", " ", regex=True)
    .str.replace(r"(?i),? (?:Jr\.?|Sr\.?|II|III|IV)$", "", regex=True)
)

sfsu_df["title"] = (
    sfsu_df["title"]
    .str.split(r"\s+in\s+", n=1, regex=True)
    .str[0]
    .str.strip()
)
sfsu_df["title"] = sfsu_df["title"].replace(
    "Estuary and Ocean Science Center Research Professor, Adjunct Professor",
    "Adjunct Professor"
)

sfsu_output = Path(sfsu_data).with_name("sfsu_cleaned.parquet")
sfsu_df.to_parquet(sfsu_output, index=False)
sfsu_df.head(5)

,name,title,department,school,start_date
0,Miguel Abad,Assistant Professor,Child and Adolescent Development,San Francisco State University,2022
1,Bárbara Abadía-Rexach,Associate Professor,Latina/Latino Studies,San Francisco State University,2020
2,Shadee Abdi,Associate Professor,Communication Studies,San Francisco State University,2019
3,Rabab Abdulhadi,Associate Professor,Race and Resistance Studies,San Francisco State University,2007
4,Priyanvada Abeywickrama,Professor,English Language and Literature,San Francisco State University,2007


In [111]:
sfsu_df.shape

(687, 5)

In [73]:
humboldt_df = pd.read_parquet(humboldt_data)
humboldt_df.head(5)

,name,school,department,title,major
0,AJ Murphy,Cal Poly Humboldt,"Forest, Fire, Range",Lecturer,<NA>
1,Aaron Donaldson,Cal Poly Humboldt,Communication,Lecturer,<NA>
2,Aaron Hohl,Cal Poly Humboldt,"Forest, Fire, Range",Lecturer,<NA>
3,Aaron Phelps,Cal Poly Humboldt,NorCal SBDC,Marketing Director,<NA>
4,Aaron Sweat,Cal Poly Humboldt,FacMgt- Plan/Des/Con,Senior Designer,<NA>


In [74]:
allowed_titles = [
    "Assistant Professor",
    "Teacher",
    "Associate Professor",
    "Professor",
    "Lecturer",
    "Lecturer AY",
    "Instructional Faculty",
    "Instructor",
    "Chair",
    "Director",
    "Grsf If 12",
    "Adjunct Professor",
    "Associate Librarian",
    "Librarian",
    "Adjunct Assistant Professor",
    "Associate Clinical Professor",
    "Senior Lecturer",
    "Full-Time Lecturer",
    "Full-time Temporary",
]

humboldt_df = humboldt_df[humboldt_df["title"].isin(allowed_titles)].copy()
humboldt_output = Path(humboldt_data).with_name("humboldt_cleaned.parquet")
humboldt_df.to_parquet(humboldt_output, index=False)
humboldt_df.head(5)


,name,school,department,title,major
0,AJ Murphy,Cal Poly Humboldt,"Forest, Fire, Range",Lecturer,<NA>
1,Aaron Donaldson,Cal Poly Humboldt,Communication,Lecturer,<NA>
2,Aaron Hohl,Cal Poly Humboldt,"Forest, Fire, Range",Lecturer,<NA>
6,Abigail Queen,Cal Poly Humboldt,History,Lecturer,<NA>
7,Abigail Smithson,Cal Poly Humboldt,Art + Film,Lecturer,<NA>


In [110]:
humboldt_df.shape

(586, 5)

In [87]:
sonoma_df = pd.read_parquet(sonoma_data)
sonoma_df.head(5)

,name,school,department,title,major
0,Aaron Westman,Sonoma State University,Department of Music,Special Consultant - Hourly,<NA>
1,Aaron Westman,Sonoma State University,Music Studio,Lecturer AY,<NA>
2,Abbie Page,Sonoma State University,Office of Student Involvement,Fraternity and Sorority Life Advisor,<NA>
3,Adam Hill,Sonoma State University,Department of Counseling,Instructional Faculty,<NA>
4,Adam Jarman,Sonoma State University,University Development,Associate Vice President for Development,<NA>


In [89]:
sonoma_df.shape

(392, 5)

In [88]:
sonoma_allowed_titles = [
    "Lecturer AY",
    "Instructional Faculty",
    "Instructor",
    "Chair",
    "Director",
    "Grsf If 12",
    "Instr Fac,Extension Non-Credit",
    "Instr Fac,Spcl Pgms-For Credit",
    "Instr Fac 12 Mo",
    "Grnt Rltd Spc Fnd Inst Fac AY",
    # Include chairs and librarians, consistent with your broader list:
    "Dept Chair AY",
    "Dept Chair 12 Mo",
    "Librarian 12 Mo",
]
sonoma_df = sonoma_df[sonoma_df["title"].isin(sonoma_allowed_titles)].copy()
sonoma_output = Path(sonoma_data).with_name("sonoma_cleaned.parquet")
sonoma_df.to_parquet(sonoma_output, index=False)
sonoma_df.head(5)

,name,school,department,title,major
1,Aaron Westman,Sonoma State University,Music Studio,Lecturer AY,<NA>
3,Adam Hill,Sonoma State University,Department of Counseling,Instructional Faculty,<NA>
6,Adam Zagelbaum,Sonoma State University,Department of Counseling,Instructional Faculty,<NA>
7,Adele Santana,Sonoma State University,Business Administration,Instructional Faculty,<NA>
14,Aidong Hu,Sonoma State University,Business Administration,Instructional Faculty,<NA>


In [109]:
sonoma_df.shape


(392, 5)

In [97]:
pomona_df = pd.read_parquet(pomona_data)
pomona_df.head(5)

,name,school,department,title,major
0,Aaron F. Cayer,Cal Poly Pomona,Architecture,Associate Professor,<NA>
1,Aaron F. Fox,Cal Poly Pomona,Department of Plant Science,Associate Professor,<NA>
2,Aaron M DeRosa,Cal Poly Pomona,English and Modern Languages,Assistant Professor,<NA>
3,Abdelfattah Amamra,Cal Poly Pomona,Computer Science,Associate Professor,<NA>
4,Abhishek Tiwari,Cal Poly Pomona,Urban and Regional Planning,<NA>,<NA>


clean middle initial

In [ ]:
pomona_df["name"] = (
    pomona_df["name"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.replace(r"(?<=\S) (?:[A-Za-z]\.? )+(?=\S)", " ", regex=True)
    .str.replace(r"(?i),? (?:Jr\.?|Sr\.?|II|III|IV)$", "", regex=True)
)


,name,school,department,title,major
0,Aaron Cayer,Cal Poly Pomona,Architecture,Associate Professor,<NA>
1,Aaron Fox,Cal Poly Pomona,Department of Plant Science,Associate Professor,<NA>
2,Aaron DeRosa,Cal Poly Pomona,English and Modern Languages,Assistant Professor,<NA>
3,Abdelfattah Amamra,Cal Poly Pomona,Computer Science,Associate Professor,<NA>
4,Abhishek Tiwari,Cal Poly Pomona,Urban and Regional Planning,<NA>,<NA>


In [92]:
pomona_df['title'].unique().tolist()

['Associate Professor',
 'Assistant Professor',
 <NA>,
 'Emeritus Professor, Advisor',
 'Adjunct Professor',
 'Professor',
 'Professor, Director of The Democracy Institute',
 'Lecturer',
 'Professor and Acting Chair',
 'Professional Practice Professor',
 'Associate Professor of Applied Linguistics/TESOL',
 'Professor Emeritus',
 'Chair and Professor',
 'Associate Professor of Anthropology',
 'Professor Emerita',
 'Professor of Biological Sciences',
 'Adjunct Faculty/Lecturer',
 'Sociology Professor',
 'Assistant Professor of Anthropology',
 'Associate Professor/Interim Chair',
 'Professor and Program Director for Sociology',
 'Assistant Professor of Environmental and Science Education',
 'Chair, Associate Professor',
 'Adjunct Professor of Engineering',
 'Program Director',
 'Basketball Coach',
 'Professor Emerita, Nutrition',
 'MPA Director',
 'Associate Professor & Production Manager',
 'Visiting Faculty',
 'Professor and Department Chair, Economics',
 'Title',
 'Professor of Organiz

In [103]:
titles = (
    pomona_df["title"]
    .astype("string")
    .str.strip()
)

pattern = (
    r"(?i)\b("
    r"Senior Assistant Librarian|"
    r"Assistant Professor|"
    r"Associate Professor|"
    r"Professor|"
    r"Adjunct Professor|"
    r"Full-Time Lecturer|"
    r"Lecturer|"
    r"Librarian"
    r")\b"
)

matched = titles.str.extract(pattern, expand=False)
pomona_df["title_clean"] = matched.str.title().fillna(titles)
# Keep unmatched titles for review instead of dropping them.
pomona_df["title"] = pomona_df["title_clean"].fillna(pomona_df["title"])
pomona_df.drop(columns=["title_clean"], inplace=True)
allowed_titles = [
    "Assistant Professor",
    "Teacher",
    "Associate Professor",
    "Professor",
    "Lecturer",
    "Lecturer AY",
    "Instructional Faculty",
    "Instructor",
    "Chair",
    "Director",
    "Grsf If 12",
    "Adjunct Professor",
    "Associate Librarian",
    "Librarian",
    "Adjunct Assistant Professor",
    "Associate Clinical Professor",
    "Senior Lecturer",
    "Full-Time Lecturer",
    "Full-time Temporary",
]

pomona_df = pomona_df[pomona_df["title"].isin(allowed_titles)].copy()

pomona_output = Path(pomona_data).with_name("pomona_cleaned.parquet")
pomona_df.to_parquet(pomona_output, index=False)
pomona_df.head(5)

,name,school,department,title,major
0,Aaron Cayer,Cal Poly Pomona,Architecture,Associate Professor,<NA>
1,Aaron Fox,Cal Poly Pomona,Department of Plant Science,Associate Professor,<NA>
2,Aaron DeRosa,Cal Poly Pomona,English and Modern Languages,Assistant Professor,<NA>
3,Abdelfattah Amamra,Cal Poly Pomona,Computer Science,Associate Professor,<NA>
5,Abigail Clarke,Cal Poly Pomona,Department of Nutrition and Food Science,Assistant Professor,<NA>


In [108]:
pomona_df.shape

(817, 5)

In [25]:
sanbern_df = pd.read_parquet(sanbern_data)
sanbern_df.head(10)

,name,school,department,title,major
0,AJ Hernandez,CSU San Bernardino,Academic Affairs - Office of the Vice President,Administrative Support Coordinator,<NA>
1,Aaron Bookman,CSU San Bernardino,Facilities Services - Grounds,Light Automotive Equipment Operator,<NA>
2,Aaron Brown,CSU San Bernardino,Ethnic Studies,Assistant Professor,<NA>
3,Abbas Quamar,CSU San Bernardino,Special Education Rehabilitation and Counseling,Professor,<NA>
4,Abdul Motin Howlader,CSU San Bernardino,"Computer Science and Engineering, School of",Lecturer,<NA>
5,Abigail Arguijo,CSU San Bernardino,Student Health Center,Pharmacist,<NA>
6,Abigail Mutya,CSU San Bernardino,Nursing,Lecturer,<NA>
7,Abigail Spivey,CSU San Bernardino,Coyote Bookstore,<NA>,<NA>
8,Acquillahs Muteti,CSU San Bernardino,Teacher Education and Foundation,Lecturer,<NA>
9,Ada Fung,CSU San Bernardino,Special Education Rehabilitation and Counseling,Lecturer,<NA>


In [105]:
sanbern_df['title'].unique().tolist()

['Administrative Support Coordinator',
 'Light Automotive Equipment Operator',
 'Assistant Professor',
 'Professor',
 'Lecturer',
 'Pharmacist',
 <NA>,
 'Admissions Processor',
 'Director, Community Engagement and Philanthropy, Palm Desert Campus',
 'Athletic Trainer',
 'Systems Administrator',
 'Groundsworker',
 'Human Resources Professional',
 'Admissions Counselor / Recruiter',
 'Teacher',
 'Philanthropic Grants Research and Development Professional',
 "Interim Associate Vice President, Student Equity and Director of Veteran's Center",
 'Administrative Analyst, Strategic Enrollment Management & Marketing',
 'Master Teacher',
 'Plumber',
 'Senior Media Relations Specialist',
 'Keyboard Collaborative Artist',
 'Associate Director for Housing Facilities',
 'Program Administrative Coordinator',
 'Custodian',
 'Director',
 'Exhibition Preparator',
 'Community Manager Lead Content Developer',
 'Senior Budget Analyst',
 'Business Systems Analyst',
 'Administrative Analyst/Specialist',
 'Li

In [106]:
titles = (
    sanbern_df["title"]
    .astype("string")
    .str.strip()
)

pattern = (
    r"(?i)\b("
    r"Senior Assistant Librarian|"
    r"Assistant Professor|"
    r"Associate Professor|"
    r"Professor|"
    r"Adjunct Professor|"
    r"Full-Time Lecturer|"
    r"Lecturer|"
    r"Librarian"
    r")\b"
)

matched = titles.str.extract(pattern, expand=False)
sanbern_df["title_clean"] = matched.str.title().fillna(titles)
# Keep unmatched titles for review instead of dropping them.
sanbern_df["title"] = sanbern_df["title_clean"].fillna(sanbern_df["title"])
sanbern_df.drop(columns=["title_clean"], inplace=True)
allowed_titles = [
    "Assistant Professor",
    "Teacher",
    "Associate Professor",
    "Professor",
    "Lecturer",
    "Lecturer AY",
    "Instructional Faculty",
    "Instructor",
    "Chair",
    "Director",
    "Grsf If 12",
    "Adjunct Professor",
    "Associate Librarian",
    "Librarian",
    "Adjunct Assistant Professor",
    "Associate Clinical Professor",
    "Senior Lecturer",
    "Full-Time Lecturer",
    "Full-time Temporary",
]

sanbern_df = sanbern_df[sanbern_df["title"].isin(allowed_titles)].copy()

sanbern_output = Path(sanbern_data).with_name("sanbern_cleaned.parquet")
sanbern_df.to_parquet(sanbern_output, index=False)
sanbern_df.head(5)

,name,school,department,title,major
2,Aaron Brown,CSU San Bernardino,Ethnic Studies,Assistant Professor,<NA>
3,Abbas Quamar,CSU San Bernardino,Special Education Rehabilitation and Counseling,Professor,<NA>
4,Abdul Motin Howlader,CSU San Bernardino,"Computer Science and Engineering, School of",Lecturer,<NA>
6,Abigail Mutya,CSU San Bernardino,Nursing,Lecturer,<NA>
8,Acquillahs Muteti,CSU San Bernardino,Teacher Education and Foundation,Lecturer,<NA>


In [107]:
sanbern_df.shape

(1165, 5)

In [115]:
dh_df = pd.read_parquet(dh_data)
dh_df.head(10)

,name,title,department,school,year
0,Yann Abdourazakou,Professor of Management and Marketing,Management and Marketing,"California State University, Dominguez Hills",2015
1,Jennifer Adeva,"Head Coach, Volleyball",,"California State University, Dominguez Hills",2011
2,Benyamin Ahmadnia,Assistant Professor of Computer Science,Computer Science,"California State University, Dominguez Hills",2024
3,Adekunle Aina,Assistant Professor of Physics,Physics,"California State University, Dominguez Hills",2024
4,Adriana Aldana,Associate Professor of Social Work,Social Work,"California State University, Dominguez Hills",2015
5,Amy Allen,Associate Professor of Dance,Dance,"California State University, Dominguez Hills",2018
6,Cathryn Andrade,Assistant Professor of Nursing,Nursing,"California State University, Dominguez Hills",2018
7,Terri Ares,Associate Professor of School of Nursing,School of Nursing,"California State University, Dominguez Hills",2015
8,Nallely Arteaga,Assistant Professor of Teacher Education,Teacher Education,"California State University, Dominguez Hills",2020
9,Hugo Asencio,Professor of Public Administration,Public Administration,"California State University, Dominguez Hills",2013


In [116]:
# Replace df with your dataframe name.
titles = (
    dh_df["title"]
    .astype("string")
    .str.strip()
    .str.replace(r"\bAssoiciate\b", "Associate", regex=True, case=False)
    .str.replace(r"\bProfesssor\b", "Professor", regex=True, case=False)
)

# Put more specific titles before their shorter versions.
pattern = (
    r"(?i)\b("
    r"Senior Assistant Librarian|"
    r"Assistant Professor|"
    r"Associate Professor|"
    r"Professor|"
    r"Full-Time Lecturer|"
    r"Lecturer|"
    r"Librarian"
    r")\b"
)

matched = titles.str.extract(pattern, expand=False)
dh_df["title_clean"] = matched.str.title().fillna(titles)
# Keep unmatched titles for review instead of dropping them.
dh_df["title"] = dh_df["title_clean"].fillna(dh_df["title"])
dh_df["title"] = dh_df["title"].replace(
    "AssociateProfessor of Physical Therapy",
    "Associate Professor"
)
dh_df.drop(columns=["title_clean"], inplace=True)
dh_df.head(10)

,name,title,department,school,year
0,Yann Abdourazakou,Professor,Management and Marketing,"California State University, Dominguez Hills",2015
1,Jennifer Adeva,"Head Coach, Volleyball",,"California State University, Dominguez Hills",2011
2,Benyamin Ahmadnia,Assistant Professor,Computer Science,"California State University, Dominguez Hills",2024
3,Adekunle Aina,Assistant Professor,Physics,"California State University, Dominguez Hills",2024
4,Adriana Aldana,Associate Professor,Social Work,"California State University, Dominguez Hills",2015
5,Amy Allen,Associate Professor,Dance,"California State University, Dominguez Hills",2018
6,Cathryn Andrade,Assistant Professor,Nursing,"California State University, Dominguez Hills",2018
7,Terri Ares,Associate Professor,School of Nursing,"California State University, Dominguez Hills",2015
8,Nallely Arteaga,Assistant Professor,Teacher Education,"California State University, Dominguez Hills",2020
9,Hugo Asencio,Professor,Public Administration,"California State University, Dominguez Hills",2013


In [118]:
dh_df.shape
dh_output = Path(sanbern_data).with_name("csudh_cleaned.parquet")
dh_df.to_parquet(dh_output, index=False)
dh_df.head(5)

,name,title,department,school,year
0,Yann Abdourazakou,Professor,Management and Marketing,"California State University, Dominguez Hills",2015
1,Jennifer Adeva,"Head Coach, Volleyball",,"California State University, Dominguez Hills",2011
2,Benyamin Ahmadnia,Assistant Professor,Computer Science,"California State University, Dominguez Hills",2024
3,Adekunle Aina,Assistant Professor,Physics,"California State University, Dominguez Hills",2024
4,Adriana Aldana,Associate Professor,Social Work,"California State University, Dominguez Hills",2015
